# Chapter 12 &mdash; A PDA for $L_{Dyck}$, and its Simulation

**Concept 3 of the Chapter 12 decomposition:** *A PDA for $L_{Dyck}$, and its Simulation*

Push on `(`, pop on `)`, accept when the input is gone and `#` is on top.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-PDA-For-Dyck/Concept-PDA-For-Dyck.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.AnimatePDA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimatePDA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimatePDA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The design, in three lines of reasoning:

* an unmatched `(` is a **debt** &mdash; push a marker;
* a `)` **pays** a debt &mdash; pop a marker; if there is nothing to pop the run dies,
  which is exactly the **prefix condition**;
* at the end the stack must hold **only** `#` &mdash; that is the **count condition**.

The two Dyck conditions of Chapter 11 map one-to-one onto two features of the machine.
That correspondence is the point of the example.

Simulate it and watch the stack height trace the same hill/valley curve as the plot in
Chapter 11, Concept 8.

## 2. Definitions

### The machine

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
Dyck = md2mc('''PDA
!! Push on '(', pop on ')', accept when the input is gone and # is on top.
I : ( , #  ; (#  -> I     !! first '(' -- push it above the bottom marker
I : ( , (  ; ((  -> I     !! another '(' -- push
I : ) , (  ; ''  -> I     !! ')' matches -- pop
I : '' , # ; #   -> F     !! nothing left and stack is just # -- accept
''')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

### A hand simulation, tracking the stack

In [ ]:
def simulate(s):
    st = ['#']
    print("  start          stack %s" % ''.join(st))
    for ch in s:
        if ch == '(':
            st.append('(')
        else:
            if st[-1] != '(':
                print("  read ')'       stack %s  <- nothing to pop: DIE" % ''.join(st))
                return False
            st.pop()
        print("  read '%s'       stack %s" % (ch, ''.join(st)))
    ok = st == ['#']
    print("  end            stack %s  -> %s" % (''.join(st), "ACCEPT" if ok else "reject"))
    return ok

<!-- nav-strip -->

---

&larr;&nbsp;[Ch12&nbsp;2.&nbsp;The PDA Edge Label: `input , pop ; push`](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-PDA-Edge-Label/Concept-PDA-Edge-Label.ipynb) &nbsp;&middot;&nbsp; [**Chapter 12** index](https://github.com/ganeshutah/Jove/blob/master/Chapter12-PDA/README.md) &nbsp;&middot;&nbsp; [Ch12&nbsp;4.&nbsp;The Formal PDA $(Q,\Sigma,\Gamma,\Delta,q_0,z_0,F)$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-Formal-PDA/Concept-Formal-PDA.ipynb)&nbsp;&rarr;

---

## 3. Tests

A successful run.

In [ ]:
ok = simulate('(())')
assert ok == pda_accepts(Dyck, '(())', STKMAX=8)

The **prefix condition** is the pop that finds nothing.

In [ ]:
ok = simulate('())')
assert not ok and not pda_accepts(Dyck, '())', STKMAX=8)

The **count condition** is the stack not being clean at the end.

In [ ]:
ok = simulate('(()')
assert not ok and not pda_accepts(Dyck, '(()', STKMAX=8)

Stack height traces the same curve as Chapter 11's hill/valley plot.

In [ ]:
def heights(s):
    h, out = 0, [0]
    for ch in s:
        h += 1 if ch == '(' else -1
        out.append(h)
    return out
for s in ['(())', '()()', '(()())']:
    print("  %-9s heights %s" % (s, heights(s)))
print("\nnever negative, ends at 0 -- exactly the two Dyck conditions.")

Agreement with the specification, exhaustively.

In [ ]:
from itertools import product
def balanced(s):
    d = 0
    for ch in s:
        d += 1 if ch == '(' else -1
        if d < 0: return False
    return d == 0
strs = [''.join(p) for k in range(7) for p in product('()', repeat=k)]
bad = [s for s in strs if pda_accepts(Dyck, s, STKMAX=9) != balanced(s)]
print("mismatches over %d strings :" % len(strs), bad)
assert not bad

## 4. Animation

The Dyck PDA. Step through `(()())` and watch the debts accumulate and clear.

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(Dyck, FuseEdges=True)

## 5. Exercises


1. Modify it for two bracket kinds. How many stack symbols do you need?
2. Which single line enforces the prefix condition?
3. Rewrite it to accept by **empty stack** rather than final state.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 253 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter12-PDA/Concept-PDA-For-Dyck')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')